## New notebook to read output of MD simulation, using Adios2 library

### part 1: **print information summary + Read variables**

- Reading one of the sample output files (.bp), step by step.  
- Printing available variables and attributes, as well as their structure.

In [ ]:
output_dir = "/home/hadis/custom_vector/buildParticleOriented/buildVS/feb12_phaseOr_newOrderParam_LJphi1/outputs/"


In [ ]:
import numpy as np
from adios2 import Stream
import os

def print_output_structure(outputDir, print_summary=True, index=0):
    attributes = {}
    variables = {}
    
    filename = os.path.join(outputDir, f"run_{index}.bp")

    with Stream(filename, "r") as s:
        for i, _ in enumerate(s.steps()):
            if i == 0:
                for attr in s.available_attributes():
                    attributes[attr] = s.read_attribute(attr)
            for var in s.available_variables():
                if var not in variables:
                    variables[var] = []
                variables[var].append(s.read(var))

    for var in variables:
        variables[var] = np.array(variables[var])

    if print_summary:
        print("Attributes Summary:")
        print("-------------------")
        for idx, (name, value) in enumerate(attributes.items(), start=1):
            print(f"{idx}. {name:<40} {value}")
        print("\nVariables Summary:")
        print("------------------")
        for idx, (name, arr) in enumerate(variables.items(), start=1):
            shape = arr.shape
            if arr.ndim == 1:
                description = shape[0]
            else:
                description = f"{shape[0]} * {list(shape[1:])}"
            print(f"{idx}. {name:<35} {description}")

    return {"attributes": attributes, "variables": variables}

result = print_output_structure(output_dir, print_summary=True)


In [ ]:
import os
import re
import numpy as np
from adios2 import FileReader

def read_variable(output_dir, variable_name):

    pattern = re.compile("run_([0-9]+).bp")
    run_numbers = [int(pattern.match(x)[1]) for x in os.listdir(output_dir) if pattern.match(x)]
    
    if not run_numbers:
        raise ValueError(f"No run files found in {output_dir}")
    
    run_numbers.sort()
    variable_data = []
    temperatures = []
    
    for i, run_num in enumerate(run_numbers):
        file_path = os.path.join(output_dir, f"run_{run_num}.bp")
        with FileReader(file_path) as reader:
            if variable_name in reader.available_variables():
                var_info = reader.available_variables()[variable_name]
                steps = int(var_info.get("AvailableStepsCount", 1))
    
                data = reader.read(variable_name, step_selection=[0, steps])
                if (data.ndim)== 1:
                    data = np.reshape(data, (1*steps, data.shape[0]//steps))
                else:
                    data = np.reshape(data, (1*steps, data.shape[0]//steps, data.shape[1]))

                variable_data.extend(data)
                
                temp_label = reader.read_attribute("temperature")
                temp_label = temp_label.flatten()
                temperatures.append(temp_label)
                
            else:
                raise ValueError(f"Variable {variable_name} not found in {file_path}")
    
    print(f'read {variable_name} successfully!')        
    return {'data':np.array(variable_data), 'temperature label':np.array(temperatures)}


In [ ]:
positions=read_variable(output_dir, 'positions')
temperature=read_variable(output_dir, 'real temperature')
com_velocity=read_variable(output_dir, 'center of mass velocity')
kinetic_energy=read_variable(output_dir,'kinetic energy')
potential_energy=read_variable(output_dir,'potential energy')
orientational_order=read_variable(output_dir,'orientational order')
order_parameter=read_variable(output_dir,'order parameter')
number_of_neighbors=read_variable(output_dir,'number of neighbors')

---

### Part 2: **Plot system properties**:
- Energy Evolution
- Temperature Evolution
- Number of Neighbors
- Order Parameter
- Center of Mass Velocity

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_temperature(m_temperatures):
    temperature_data = m_temperatures['data']
    temp_labels = m_temperatures['temperature label']
    fig, ax = plt.subplots(figsize=(7, 4))
    
    ax.grid(True, linestyle='--', alpha=0.7, color='gray')

    colors = plt.cm.plasma(np.linspace(0, 1, 4))
    axis_labels = ['x', 'y', r'$\omega$']

    for i, axis_label in enumerate(axis_labels):
        ax.plot(temperature_data[:, :, i], label=f"T{axis_label}", color=colors[i])

    total_points = temperature_data.shape[0]
    tick_indices = np.linspace(0, total_points-1, 10, dtype=int)
    label_indices = np.linspace(0, len(temp_labels)-1, 10, dtype=int)

    ax.set_xlabel("Temperature Labels")
    ax.set_xticks(tick_indices)
    ax.set_xticklabels([f"T={temp_labels[i][0]:.1f}" for i in label_indices], rotation=45)
    ax.set_ylabel("Measured Temperature")
    ax.set_title("Temperature Evolution")
    
    ax.legend(bbox_to_anchor=(1, 1), loc='upper left', frameon=True, fancybox=True, shadow=False)
    
    plt.tight_layout()
    plt.show()

def plot_neighbors(neighbors_array, m_temperatures, temperature_show=False):
    neighbors_data = neighbors_array['data']
    temp_labels = neighbors_array['temperature label']
    fig, ax = plt.subplots(figsize=(7, 4))

    ax.grid(True, linestyle='--', alpha=0.7, color='gray')

    colors = plt.cm.viridis(np.linspace(0, 1, 4))

    for i in range(neighbors_data.shape[2]):
        ax.plot(neighbors_data[:, :, i], label=f"Shell {i}", color=colors[i], linewidth=2)

    total_points = neighbors_data.shape[0]
    tick_indices = np.linspace(0, total_points - 1, 10, dtype=int)
    label_indices = np.linspace(0, len(temp_labels) - 1, 10, dtype=int)

    ax.set_xticks(tick_indices)
    ax.set_xticklabels([f"T={temp_labels[i][0]:.1f}" for i in label_indices], rotation=45)

    ax.set_ylabel("Number of Neighbors")
    ax.set_title("Neighbor Analysis")
    
    ax.legend(bbox_to_anchor=(1, 1), loc='upper left', frameon=True, fancybox=True, shadow=False)

    plt.tight_layout()
    plt.show()

    if temperature_show:
        plot_temperature(m_temperatures)

def plot_energies(kinetic_energy, potential_energy, m_temperatures, show_potential=True, temperature_show=False):
    kinetic_energy_data = kinetic_energy['data']
    temp_labels = kinetic_energy['temperature label']
    potential_energy_data = potential_energy['data'] / 100

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.grid(True, linestyle='--', alpha=0.7, color='gray')

    colors = plt.cm.plasma(np.linspace(0, 1, kinetic_energy_data.shape[2] + 3))
    axis_labels = ['x', 'y', r'$\omega$']

    for i, axis_label in enumerate(axis_labels):
        ax.plot(kinetic_energy_data[:, :, i], label=f"K{axis_label}", color=colors[i])

    if show_potential:
        y2 = potential_energy_data.flatten()
        ax.plot(y2, label='U', color=colors[3])
        ax.plot(y2 + np.sum(kinetic_energy_data, axis=2)[:, 0], label='Total', color=colors[4])

    total_points = kinetic_energy_data.shape[0]
    tick_indices = np.linspace(0, total_points - 1, 10, dtype=int)
    label_indices = np.linspace(0, len(temp_labels) - 1, 10, dtype=int)

    ax.set_xticks(tick_indices)
    ax.set_xticklabels([f"T={temp_labels[i][0]:.1f}" for i in label_indices], rotation=45)

    ax.set_ylabel("Energy")
    ax.set_title("Energy Evolution")
    
    ax.legend(bbox_to_anchor=(1, 1), loc='upper left', frameon=True, fancybox=True, shadow=False)

    plt.tight_layout()
    plt.show()

    if temperature_show:
        plot_temperature(m_temperatures)

def plot_com_velocity(com_velocity, m_temperatures, temperature_show=False):
    com_vel_data = com_velocity['data']
    temp_labels = com_velocity['temperature label']
    fig, ax = plt.subplots(figsize=(7, 4))

    ax.grid(True, linestyle='--', alpha=0.7, color='gray')

    colors = plt.cm.plasma(np.linspace(0, 1, 4))
    axis_labels = ['x', 'y']

    for i, axis_label in enumerate(axis_labels):
        ax.plot(com_vel_data[:, :, i], label=f"K{axis_label}", color=colors[i])

    total_points = com_vel_data.shape[0]
    tick_indices = np.linspace(0, total_points-1, 10, dtype=int)
    label_indices = np.linspace(0, len(temp_labels)-1, 10, dtype=int)

    ax.set_xticks(tick_indices)
    ax.set_xticklabels([f"T={temp_labels[i][0]:.1f}" for i in label_indices], rotation=45)

    ax.set_ylabel("Velocity")
    ax.set_title("Center of Mass Velocity Evolution")
    
    ax.legend(bbox_to_anchor=(1, 1), loc='upper left', frameon=True, fancybox=True, shadow=False)

    plt.tight_layout()
    plt.show()

    if temperature_show:
        plot_temperature(m_temperatures)

def plot_order_parameter(order, m_temperatures, type, temperature_show=False):
    order_data = order['data']
    temp_labels = order['temperature label']
    fig, ax = plt.subplots(figsize=(6, 4))  
    
    ax.grid(True, linestyle='--', alpha=0.7, color='gray')
    ax.plot(order_data[:, :])

    total_points = order_data.shape[0]
    tick_indices = np.linspace(0, total_points-1, 10, dtype=int)
    label_indices = np.linspace(0, len(temp_labels)-1, 10, dtype=int)

    ax.set_xticks(tick_indices)
    ax.set_xticklabels([f"T={temp_labels[i][0]:.1f}" for i in label_indices], rotation=45)

    ax.set_ylabel("Order Parameter")
    ax.set_title(f"{type} Order Parameter")
        
    plt.tight_layout()
    plt.show()

    if temperature_show:
        plot_temperature(m_temperatures)


In [ ]:
plot_energies(kinetic_energy, potential_energy, temperature, show_potential=True)

In [ ]:
plot_temperature(temperature)

In [ ]:
plot_neighbors(number_of_neighbors, temperature)

In [ ]:
plot_order_parameter(orientational_order, temperature, "rotational")

In [ ]:
plot_order_parameter(order_parameter, temperature, "rotational")

In [ ]:
plot_com_velocity(com_velocity, temperature)

---

### Part 3: **Trajoctory and snapshots of system at specific temperatures**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def target_temperature_index (variable_name, target_temperature):
    data = variable_name['data']
    temperature_labels = variable_name['temperature label'].flatten()
    print(f"available temperatures are from {min(temperature_labels)} to {max(temperature_labels)}.")
          
    try:
        index = np.where(temperature_labels == target_temperature)[0][0]
    except IndexError:
        print(f"Target temperature {target_temperature} not found in temperature_labels.")
        return 0

    total_snapshots = data.shape[0]
    return int(total_snapshots / len(temperature_labels) * index)

def plot_trajectory(positions, target_temperature, step_window=10000):
    positions_data = positions['data']

    step_min = target_temperature_index(positions, target_temperature)
    step_max = step_min + step_window

    fig, ax = plt.subplots(1, 2, figsize=(8, 4))

    num_particles = positions_data.shape[1]

    for i in range(num_particles):
        ax[0].scatter(
            positions_data[step_min:step_max, i, 0], 
            positions_data[step_min:step_max, i, 1], 
            c=np.linspace(0, 1, step_max - step_min),
            cmap="plasma",
            s=0.02, alpha=0.6
        )

    ax[0].set_title(f"Trajectories at T={target_temperature} for {step_window} steps")
    ax[0].set_xlabel("X Position")
    ax[0].set_ylabel("Y Position")

    # Plot initial vs final positions
    ax[1].scatter(
        positions_data[step_min, :, 0], positions_data[step_min, :, 1], 
        marker='o', s=20, color='#00aabb', label='Initial Position'
    )
    ax[1].scatter(
        positions_data[step_max - 1, :, 0], positions_data[step_max - 1, :, 1], 
        marker='x', s=20, color='#ff7777', label='Final Position'
    )

    ax[1].set_title("Initial vs. Final Positions")
    ax[1].set_xlabel("X Position")
    ax[1].set_ylabel("Y Position")
    ax[1].legend()

    for ax_i in ax:
        ax_i.set_xlim(0,20)
        ax_i.set_ylim(0,20)
        ax_i.set_aspect('equal', adjustable='datalim')
        ax_i.grid(linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()

def plot_snapshot_oriented(positions, target_temperature, color_palette='hsv'):
    positions_data = positions['data']
    shot = target_temperature_index(positions, target_temperature)

    fig, ax = plt.subplots(figsize=(6, 4))
    
    scatter = ax.scatter(
        positions_data[shot, :, 0],    # X
        positions_data[shot, :, 1],    # Y
        c=positions_data[shot, :, 2],  # φ
        s=50,
        alpha=0.8,
        vmin=-np.pi,
        vmax=np.pi,
        cmap=color_palette
    )
        
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('φ in radian')
        
    ax.set_xlabel('X Position')
    ax.set_ylabel('Y Position')
    ax.set_title(f'Positions + Orientation φ (T={target_temperature})')
    
    ax.set_xlim(0, 20)
    ax.set_ylim(0, 20)
    ax.set_aspect('equal', adjustable='box')
    ax.grid(linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()


In [ ]:
plot_snapshot_oriented(positions, 0.02,'hsv')

In [ ]:
plot_trajectory(positions, 0.002)

---

### Part 3: **Depiction of $\phi$ and $\Delta\phi$**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_hist_phi(positions, target_temperature, range_read=1000, bins_num=100):
    positions_data = positions['data']

    shot = target_temperature_index(positions, target_temperature)

    fig, ax = plt.subplots(figsize=(6, 4))

    ax.hist(positions_data[shot-range_read:shot+range_read,:,2].flatten(), bins=bins_num)
    ax.set_title(f'Histogram of particles $\phi$ over {range_read} steps, at T = {target_temperature}')
    ax.set_xlabel('Orientation angle $\phi$ (rad)')
    ax.set_ylabel('Frequency')
    ax.set_axisbelow(True)
    ax.grid(color='gray', linestyle='dashed', alpha=0.5)
    
    plt.tight_layout()
    plt.show()


In [ ]:
plot_hist_phi(positions, 0.02)

NOte: Do to list:

1. delta phi over distances (hist , delta phi, distances)
    - closer than a specific distance, plot deltaphi..
    - 3D plot of deltaphi histogram
2. lower temperature

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_delta_phi_hist(positions, target_t, min_dis, bins_num= 100):
    index = target_temperature_index(positions, target_t)
    data = positions['data'][index]

    total_delta_phi = []

    for i in range(data.shape[0]):
        for j in range(i+1, data.shape[0]):
            r = np.sqrt((data[i, 0] - data[j, 0])**2 + (data[i, 1] - data[j, 1])**2)
            if r < min_dis:
                total_delta_phi.append(data[i, 2] - data[j, 2])

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(total_delta_phi, bins=bins_num, color='#55b3d1')

    ax.set_xlabel(r'$\Delta \phi$ (rad)')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Histogram of $\Delta \phi$ at T={target_t}')
    ax.set_axisbelow(True)
    ax.grid(color='gray', linestyle='dashed', alpha=0.5)

    plt.show()


In [ ]:
plot_delta_phi_hist(positions, 0.02, 10)

---

### Part 4: **Animation of particles movement**

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.cm as cm


def map_to_frame(x, y, frame_size, offset=0):
    scale = frame_size / 20
    return int(x * scale + offset), int(y * scale + offset)

def phi_to_color(phi):
    norm_phi = phi / (2 * np.pi) 
    color = cm.twilight(norm_phi)
    bgr_color = (int(color[2] * 255), int(color[1] * 255), int(color[0] * 255))
    return bgr_color

def temperature_to_color(temp):
    norm_temp = (temp - 0.1) / (1 - 0.1)
    color = cm.coolwarm(norm_temp)
    bgr_color = (int(color[2] * 255), int(color[1] * 255), int(color[0] * 255))
    return bgr_color
    
def animation_particles(file, particle_type, skip_rows, output_name, phase_transition = False) :
    frame_size = 800
    particle_radius = 10

    type_size = 2 if particle_type == "dot" else 3

    positions = []
    with open(file, 'r') as file:
        for i, line in enumerate(file):
            if i % skip_rows == 0:
                data = line.strip().split()
                data = list(map(float, data[1:]))
                positions.append(data)
    positions = np.array(positions)

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_name, fourcc, 10.0, (frame_size + 200, frame_size))

    high_temp = 1.0
    low_temp = 0.1
    num_frames = len(positions)
    temperatures = np.linspace(high_temp, low_temp, num_frames // 2).tolist() + np.linspace(low_temp, high_temp, num_frames - num_frames // 2).tolist()

    for i, frame_data in enumerate(positions):
        frame = np.ones((frame_size, frame_size + 200, 3), dtype=np.uint8) * 255

        # Draw bounding box around particles
        cv2.rectangle(frame, (0, 0), (frame_size, frame_size), (0, 0, 0), 2)

        # Draw particles
        for j in range(0, len(frame_data), type_size):
            x, y = frame_data[j], frame_data[j + 1]
            cx, cy = map_to_frame(x, y, frame_size)

            if type_size == 3:
                phi = frame_data[j + 2] + np.pi
                color = phi_to_color(phi)
            else:
                color = (120, 120, 120)

            cv2.circle(frame, (cx, cy), particle_radius, color, -1)
        if phase_transition == True :
            # Draw temperature bar with gradient
            bar_x_start = frame_size + 50
            bar_x_end = frame_size + 150
            bar_y_start = 50
            bar_y_end = frame_size - 50
            for y in range(bar_y_start, bar_y_end):
                t = (y - bar_y_start) / (bar_y_end - bar_y_start)
                color = cm.coolwarm(1 - t)
                bgr_color = (int(color[2] * 255), int(color[1] * 255), int(color[0] * 255))
                cv2.line(frame, (bar_x_start, y), (bar_x_end, y), bgr_color, 1)

            # Draw indicator for current temperature
            temp = temperatures[i]
            indicator_y = int(bar_y_end - (temp - 0.1) / (1 - 0.1) * (bar_y_end - bar_y_start))
            cv2.arrowedLine(frame, (bar_x_end + 10, indicator_y), (bar_x_end + 50, indicator_y), (0, 0, 0), 2, tipLength=0.3)

        out.write(frame)

        if i % 50 == 0:
            print(f"Processing frame {i}/{len(positions)}")

    out.release()
    print("Video saved as", output_name)

    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.title("Particle Movement at Last Frame")
    plt.show()